In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

## Passo 1: dados v3 (a receita da T4)

In [2]:
%%bash
python src/get_data.py --until 2012-12-31
dvc add data/raw/bike.csv
git add data/raw/bike.csv.dvc
git commit -m "Dados v3: 2012 completo"
git tag dados-v3
dvc push

data/raw/bike.csv: 17,379 linhas | 2011-01-01 a 2012-12-31


⠋ Checking graph



To track the changes with git, run:

	git add data/raw/bike.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
On branch main
Your branch is ahead of 'origin/main' by 7 commits.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   dvc.lock
	modified:   dvc.yaml
	modified:   metrics.json
	modified:   params.yaml
	modified:   reports/previsto_vs_real.png

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	aula01/t0_ambiente.ipynb
	aula01/t1_pipeline_dvc.ipynb
	aula01/t2_mlflow_tracking.ipynb
	aula01/t3_model_registry.ipynb
	aula01/t4_dvc_dados.ipynb
	desafio01/

no changes added to commit (use "git add" and/or "git commit -a")


fatal: tag 'dados-v3' already exists


Everything is up to date.


## Passo 2: complete o `compare_models.py`

Abra `desafio01/compare_models.py` e complete os 3 `TODO`. As dicas estão no próprio arquivo.
O script avalia `@champion` e `@challenger` **no mesmo teste** e grava `reports/comparacao.json`.

### Feito

## Passo 3: escreva o estágio `compare` no `dvc.yaml`

Acrescente um 6º estágio no fim do `dvc.yaml`, no mesmo molde dos estágios da T1:

| Chave | O que colocar |
|---|---|
| `cmd` | `python desafio01/compare_models.py` |
| `deps` | o próprio script, a pasta `models` e `data/split/test.csv` |
| `metrics` | `reports/comparacao.json`, com `cache: false` (igual ao `metrics.json` do `evaluate`) |

Confira com `dvc dag`: o `compare` deve aparecer depois do `train`.


In [3]:
%%writefile dvc.yaml
# Pipeline do projeto. Cada estágio declara o comando (cmd), o que ele lê (deps),
# os parâmetros do params.yaml que usa (params) e o que ele gera (outs).
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
    - src/prepare.py
    - src/bike.py
    - data/raw/bike.csv
    outs:
    - data/prepared

  split:
    cmd: python src/split.py
    deps:
    - src/split.py
    - src/bike.py
    - data/prepared
    params:
    - split
    outs:
    - data/split

  train:
    cmd: python src/train.py
    deps:
    - src/train.py
    - src/bike.py
    - data/split/train.csv
    params:
    - train
    - mlflow
    outs:
    - models

  evaluate:
    cmd: python src/evaluate.py
    deps:
    - src/evaluate.py
    - src/bike.py
    - models
    - data/split/test.csv
    metrics:
    - metrics.json:
        cache: false

  plot:
    cmd: python src/plot.py
    deps:
    - src/plot.py
    - models
    - data/split/test.csv
    plots:
    - reports/previsto_vs_real.png:
        cache: false

  compare:
    cmd: python desafio01/compare_models.py
    deps:
    - desafio01/compare_models.py
    - models
    - data/split/test.csv
    metrics:
    - reports/comparacao.json:
        cache: false

Overwriting dvc.yaml


## Passo 4: retreinar e comparar

In [4]:
%%bash
dvc repro                    # prepare → split → train (nova versão no Registry) → evaluate, plot e compare
cat reports/comparacao.json
dvc metrics diff             
dvc push

'data/raw/bike.csv.dvc' didn't change, skipping
Stage 'prepare' didn't change, skipping
Stage 'split' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Stage 'plot' didn't change, skipping
Running stage 'compare':
> python desafio01/compare_models.py
            versao    rmse     mae     r2
champion       7.0  64.602  43.847  0.839
challenger     7.0  64.602  43.847  0.839

Teste: 2012-12-04 a 2012-12-31 (670 linhas)
Challenger NÃO venceu: RMSE 64.6 vs 64.6 (champion)
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
{
  "champion": {
    "versao": 7,
    "rmse": 64.60239487088145,
    "mae": 43.84721260070395,
    "r2": 0.8393715464626234
  },
  "challenger": {
    "versao": 7,
    "rmse": 64.60239487088145,
    "mae": 43.84721260070395,
    "r2": 0.8393715464626234
  },
  "teste"

## Passo 5: decidir

In [5]:
%%bash
dvc dag
dvc repro
cat reports/comparacao.json
dvc metrics diff
dvc push
python desafio01/compare_models.py --promover

               +-----------------------+                       
               | data/raw/bike.csv.dvc |                       
               +-----------------------+                       
                            *                                  
                            *                                  
                            *                                  
                      +---------+                              
                      | prepare |                              
                      +---------+                              
                            *                                  
                            *                                  
                            *                                  
                       +-------+                               
                      *| split |***                            
                 ***** +-------+   ****                        
             ****      *        *      *

## Passo 6: relatório

In [6]:
%%bash
for tag in dados-v1 dados-v2 dados-v3; do
  echo "$tag $(git show $tag:data/raw/bike.csv.dvc | awk '/md5/{print $3}')"
done

dados-v1 fc7d423afbce5699a61ac1a8acc533b8
dados-v2 11e40c60d906dcbd0ebad321fcd221e0
dados-v3 7747b2efcdd3801b83c1cfce2d600c15


## Passo 7: entrega

In [7]:
%%bash
git add dvc.yaml dvc.lock metrics.json reports/ desafio01/
git commit -m "Desafio 1: dados v3, estágio compare e decisão"
git push
git push --tags

[main f1ba303] Desafio 1: dados v3, estágio compare e decisão
 9 files changed, 792 insertions(+), 26 deletions(-)
 create mode 100644 desafio01/README.md
 create mode 100644 desafio01/RELATORIO.md
 create mode 100644 desafio01/compare_models.py
 create mode 100644 desafio01/d1-novos_dados.ipynb
 create mode 100644 reports/comparacao.json


To https://github.com/DiasDavi/mlops-facens
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/DiasDavi/mlops-facens'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.
To https://github.com/DiasDavi/mlops-facens
 * [new tag]         dados-v3 -> dados-v3
